In [ ]:
import sys
import os
from google.colab import userdata

# 1. 自动从 Colab Secrets 获取你存入的 Token
GIT_TOKEN = userdata.get('GITHUB_TOKEN')

# 2. 根据你的截图填入对应的用户名和仓库名
GIT_USER = "Sheng-Pan"
GIT_REPO = "Decentralized-federated-learning2"

# 3. 构建带认证信息的 URL
repo_url = f"https://{GIT_TOKEN}@github.com/{GIT_USER}/{GIT_REPO}.git"

# 4. 克隆仓库 (如果文件夹已存在则跳过，防止报错)
if not os.path.exists(GIT_REPO):
    !git clone {repo_url}
else:
    print(f"{GIT_REPO} already exists.")

# 5. 将仓库路径添加到系统路径，以便 Python 找到 functions2.py
repo_path = os.path.join("/content", GIT_REPO)
if repo_path not in sys.path:
    sys.path.append(repo_path)


Cloning into 'Decentralized-federated-learning2'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 43 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 139.84 KiB | 633.00 KiB/s, done.
Resolving deltas: 100% (22/22), done.


In [ ]:
# 4. 强制更新仓库 (如果存在则拉取最新，不存在则克隆)
if not os.path.exists(GIT_REPO):
    print("Cloning new repository...")
    !git clone {repo_url}
else:
    print(f"{GIT_REPO} already exists. Pulling latest changes...")
    # 切换进目录更新，然后再切换出来
    %cd {GIT_REPO}
    !git pull
    %cd ..

# 5. 关键的一步：强制重新加载已经导入的模块
import importlib
import defense
importlib.reload(defense)
import backdoor
importlib.reload(backdoor)
import MAB_fun
importlib.reload(MAB_fun)
import trainer
importlib.reload(trainer)
import eval_DFL
importlib.reload(eval_DFL)
import main_cnn_GPU
importlib.reload(main_cnn_GPU)

<module 'main_cnn_GPU' from '/content/Decentralized-federated-learning2/main_cnn_GPU.py'>

In [ ]:

import os
import time
import pandas as pd
import sys

from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity
from google.colab import userdata
import copy
from torch.utils.data import DataLoader
from huggingface_hub import HfApi, snapshot_download


original_stdout = sys.stdout
original_stderr = sys.stderr
class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()

import sys

NORM_FACTOR = 1
def upload_with_retry(path, repo_path, max_retries=3):
    for i in range(max_retries):
        try:
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=repo_path,
                repo_id=REPO_ID,
                repo_type="dataset"
            )
            print(f"✅ 上传成功: {os.path.basename(path)}")
            return True
        except Exception as upload_err:
            print(f"⚠️ 第 {i+1} 次上传失败 ({os.path.basename(path)}): {upload_err}")
            if i < max_retries - 1:
                time.sleep(5)  # 等待5秒后重试
            else:
                print(f"❌ 最终上传放弃: {os.path.basename(path)}")
                return False
HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "JONESMITH007/DFL"
LOCAL_ROOT = "./DFL"
api = HfApi(token=HF_TOKEN)
SAVE_PATH = os.path.join(LOCAL_ROOT, "FL_Experiments/cnn_Results_2026_sensitivity2")
os.makedirs(SAVE_PATH, exist_ok=True)
# ==========================================
# 1.Experiment settup
# ==========================================



NUM_CLIENTS = 20
BOOST_FACTORS = 2
MALICIOUS_RATIO = 0.3
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)


SEEDS = [1,2,3]
MAL_RATIOS = [ 0.3, 0.1, 0.2]
TOPO_TYPES = [ 'scale_free','random_regular']
DEFENSE_RATIOS = [ 0.2]
mech = 'MAB'
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0
def_ratio = 0.2
mal_ratio = 0.3
# --- 敏感性分析超参数网格 ---
AUDIT_PROBS = [0.8, 0.9]
AGG_PROBS = [0.8, 0.9]
AGG_THRE = [0.4, 0.5]

ratio = 0.3
num_mal= int(NUM_CLIENTS*ratio)
current_def_budget = int(NUM_CLIENTS*def_ratio)
for topo_type in TOPO_TYPES:
    for seed_val in SEEDS:
        for aup in AUDIT_PROBS:
            for ap in AGG_PROBS:
                for at in AGG_THRE:

                    print(f"\n{'='*60}")
                    print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"📡 : Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                    print(f"{'='*60}")
                    print(f"\n{'#'*60}")
                    print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                    print(f"{'#'*60}")

                    set_seed(seed_val)
                    client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                    G = generate_topology(NUM_CLIENTS, topo_type)
                    neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                    malicious_clients, defense_nodes = allocate_malicious_nodes(
                        G, num_mal, current_def_budget, topology_type=topo_type,placement='Topology-Aware',# placement='Topology-Aware'
                    )

                    theo_intensities = calculate_theoretical_intensity(
                        neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                    )
                    # 构造参数字符串，例如: AUP0.8_AP0.5_ST1.0
                    param_str = f"AUP{aup}_AP{ap}_AT{at}"

                    # 命名加上不同参数
                    csv_filename = f"Final_CNN_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_{param_str}_seed{seed_val}.csv"
                    log_filename = f"Final_CNN_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_{param_str}_seed{seed_val}.txt"

                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_full_path = os.path.join(SAVE_PATH, log_filename)


                    if os.path.exists(full_save_path):
                        print(f"⏩Eisting file: {csv_filename}")
                        continue
                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    print(f"🚀 Running: {mech}")
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()

                    _, _, accs, asrs = run_simulation_CNN_GPU(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False,audit_prob=aup,
                        agg_prob=ap,
                        agg_threshold=at
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)

                    result_entry_base = {
                        'seed': seed_val,
                        'mechanism': mech,
                        'audit_prob': aup,
                        'agg_prob': ap,
                         'agg_thre': at,
                        'malicious_ratio': mal_ratio,
                        'defense_ratio': def_ratio,
                        'topology': topo_type,
                        'norm_factor': NORM_FACTOR,
                        'scale_factor': SCALE_FACTOR,
                        'global_rounds': GLOBAL_ROUNDS,
                        'start_time': start_wall_time,
                        'duration_sec': duration_sec
                    }
                    all_results = []
                    for i in range(NUM_CLIENTS):
                        client_row = copy.deepcopy(result_entry_base)
                        client_row.update({
                            'client_id': i,
                            'final_acc': accs[i],
                            'final_asr': asrs[i],
                            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                        })
                        all_results.append(client_row)

                    pd.DataFrame(all_results).to_csv(full_save_path, index=False)


                    sys.stdout = original_stdout

                    csv_repo_path = f"FL_Experiments/cnn_Results_2026_sensitivity2/{csv_filename}"
                    log_repo_path = f"FL_Experiments/cnn_Results_2026_sensitivity2/{log_filename}"


                    upload_with_retry(full_save_path, csv_repo_path)
                    upload_with_retry(log_full_path, log_repo_path)

                    sys.stdout = logger
print(f"\n🎉 Experiments compeleted！")

In [ ]:
from backdoor import apply_patch_trigger_image
apply_patch_trigger_image

<function backdoor.apply_patch_trigger_image(image, intensity=1.0)>

In [ ]:
# ... [前面的 import 部分保持不变] ...
import os
import time
import pandas as pd
import torch
import gc
from google.colab import drive
import numpy as np
import random
import pandas as pd
from data_loader import get_medical_data_augmented
from torch.utils.data import DataLoader, TensorDataset, random_split
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification,  get_linear_schedule_with_warmup
import glob # <--- Added this line
import sys
import time
from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from defense import get_high_value_defense_nodes
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity
from data_loader import set_seed
import torch
import torch.nn as nn
import numpy as np
import copy
from torch.utils.data import DataLoader
from collections import defaultdict
import torch.nn.functional as F

from data_loader import set_seed
from MAB_fun import MABDefense_CNN,MABDefense
from MAB_fun import calculate_continuous_trust
from trainer import train_client_cnn
from defense import get_universal_stats,count_minimum_required_defenders
from defense import  calculate_krum_scores
from eval_DFL import evaluate_global_cnn
# ==========================================
# 1. CNN 专用辅助函数 (Sensitivity & Trapdoor)
# ==========================================
import sys
import os
# 保存原始输出流 (修复 original_stdout 报错)
original_stdout = sys.stdout
original_stderr = sys.stderr
class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):  # <--- 必须添加这个方法
        self.log.close()
NORM_FACTOR = 1
import sys
import os
NORM_FACTOR = 1
def upload_with_retry(path, repo_path, max_retries=3):
    for i in range(max_retries):
        try:
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=repo_path,
                repo_id=REPO_ID,
                repo_type="dataset"
            )
            print(f"✅ 上传成功: {os.path.basename(path)}")
            return True
        except Exception as upload_err:
            print(f"⚠️ 第 {i+1} 次上传失败 ({os.path.basename(path)}): {upload_err}")
            if i < max_retries - 1:
                time.sleep(5)  # 等待5秒后重试
            else:
                print(f"❌ 最终上传放弃: {os.path.basename(path)}")
                return False
HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "JONESMITH007/DFL"
LOCAL_ROOT = "./DFL"
api = HfApi(token=HF_TOKEN) # 👈 必须添加这一行
# 🔥 修正：SAVE_PATH 必须与 snapshot_download 的结构完全对应
# 如果仓库里文件夹叫 Results_transformer_Comparison，这里就不要加 FL_Experiments
SAVE_PATH = os.path.join(LOCAL_ROOT, "FL_Experiments/cnn_Results_2026_new2")
os.makedirs(SAVE_PATH, exist_ok=True)
# ==========================================
# 1. 实验设置
# ==========================================
# 假设 functions2 和 SimpleCNN 已经定义好
# from functions2 import * # from simple_cnn import SimpleCNN

# 参数配置
# BOOST_FACTORS = 2 # This is used later, no need to move
# MALICIOUS_RATIO = 0.3 # This is used later, no need to move
GLOBAL_ROUNDS = 1 # This is used later, no need to move
intensity1 = 0.2
intensity2 = 0.2
import os
# --- 3. 实验配置: 定义核心参数 (将所有 NUM_CLIENTS 相关的参数定义在一起) ---

# 定义其他固定参数 (根据你的代码片段)
NUM_CLIENTS = 20 # <--- Ensure this is set BEFORE dependent variables are created
BOOST_FACTORS = 2 # Re-declare or ensure it's defined once
MALICIOUS_RATIO = 0.3 # Re-declare or ensure it's defined once
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15 # Re-declare or ensure it's defined once
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
# --- 1. 实验扩展配置 ---

# 加载数据 (只需加载一次，放在循环外)
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)

# --- 2. 开始多维嵌套循环 ---
all_results = []
# --- 1. 实验扩展配置 ---
SEEDS = [1,2,3]
MECHANISM_LIST = ['MAB',  'FLAME','CosL2', 'TrimmedMean', 'Krum']
MECHANISM_LIST = ['MAB','FedAvg','TrimmedMean', 'Krum']
MAL_RATIOS = [ 0.3, 0.1, 0.2]
TOPO_TYPES = [ 'random_regular','scale_free']
DEFENSE_RATIOS = [ 0.2]  # 👈 新增：防御节点比例列表 (例如 10%, 20%, 30%)

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

# --- 2. 开始多维嵌套循环 ---


for topo_type in TOPO_TYPES:
    for mal_ratio in MAL_RATIOS:
        for def_ratio in DEFENSE_RATIOS: # 👈 第三层：防御比例循环

            # 动态计算当前配置
            num_mal = int(NUM_CLIENTS * mal_ratio)
            # 根据当前的 def_ratio 动态计算防御预算
            current_def_budget = int(NUM_CLIENTS * def_ratio)
            for seed_val in SEEDS:


                print(f"\n{'='*60}")
                print(f"⏰ 任务启动: {time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"📡 配置: Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                print(f"{'='*60}")
                print(f"\n{'#'*60}")
                print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                print(f"{'#'*60}")

                set_seed(seed_val)
                client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                G = generate_topology(NUM_CLIENTS, topo_type)
                neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}

                # 使用动态的 current_def_budget 分配节点
                malicious_clients, defense_nodes = allocate_malicious_nodes(
                    G, num_mal, current_def_budget, topology_type=topo_type,placement='Topology-Aware',# placement='Topology-Aware'
                )

                theo_intensities = calculate_theoretical_intensity(
                    neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                )

                for mech in MECHANISM_LIST:
                    # 1. 先定义好文件名和路径
                    csv_filename = f"Final_CNN_{topo_type}_MR{mal_ratio}_DR{def_ratio}_{mech}_seed{seed_val}.csv"
                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_filename = f"Log_CNN_{topo_type}_MR{mal_ratio}_DR{def_ratio}_{mech}_seed{seed_val}.txt"
                    log_full_path = os.path.join(SAVE_PATH, log_filename)

                    # 2. 断点续传检查 (直接检查物理文件最可靠)
                    if os.path.exists(full_save_path):
                        print(f"⏩ 跳过已存在文件: {csv_filename}")
                        continue
# 2. 增强版断点续传：直接检查物理文件是否存在
                    # 3. 启动日志 (每个方法独立日志)
                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    print(f"🚀 Running: {mech}")
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()
                    # 运行模拟
                    _, _, accs, asrs = run_simulation_CNN_GPU(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)
                    # --- 收集并保存结果 ---
                    result_entry_base = {
                        'seed': seed_val,
                        'mechanism': mech,
                        'malicious_ratio': mal_ratio,
                        'defense_ratio': def_ratio, # 👈 记录防御比例
                        'topology': topo_type,
                        'norm_factor': NORM_FACTOR,
                        'scale_factor': SCALE_FACTOR,
                        'global_rounds': GLOBAL_ROUNDS,
                        'start_time': start_wall_time,  # 👈 新增时间维度
                        'duration_sec': duration_sec    # 👈 新增耗时维度
                    }

                    for i in range(NUM_CLIENTS):
                        client_row = copy.deepcopy(result_entry_base)
                        client_row.update({
                            'client_id': i,
                            'final_acc': accs[i],
                            'final_asr': asrs[i],
                            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                        })
                        all_results.append(client_row)
                      # 保存本地
                    pd.DataFrame(all_results).to_csv(full_save_path, index=False)

                    # --- 即时上传 ---
                    sys.stdout = original_stdout
                    print(f"☁️ 正在上传结果至 HF: {csv_filename}")

                    # 执行上传逻辑
                    print(f"☁️ 准备上传结果至 HF...")
                    csv_repo_path = f"FL_Experiments/cnn_Results_2026_new2/{csv_filename}"
                    log_repo_path = f"FL_Experiments/cnn_Results_2026_new2/{log_filename}"

                    # 即使上传失败，代码也会继续往下走，不会跳入外层的 except
                    upload_with_retry(full_save_path, csv_repo_path)
                    upload_with_retry(log_full_path, log_repo_path)

                    sys.stdout = logger

print(f"\n🎉 大型实验（含防御比例维度）全部完成！")

ImportError: cannot import name 'inject_distributed_medical_trigger' from 'backdoor' (/content/Decentralized-federated-learning2/backdoor.py)

In [ ]:
# 调试专用
import os
import time
from datetime import datetime
import pytz
from torch.utils.data import DataLoader, TensorDataset, random_split
from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

# ==========================================
# 1. 实验设置
# ==========================================
# 假设 functions2 和 SimpleCNN 已经定义好
# from functions2 import * # from simple_cnn import SimpleCNN

# 参数配置
# BOOST_FACTORS = 2 # This is used later, no need to move
# MALICIOUS_RATIO = 0.3 # This is used later, no need to move
GLOBAL_ROUNDS = 1 # This is used later, no need to move
intensity1 = 0.2
intensity2 = 0.2
import os
# --- 3. 实验配置: 定义核心参数 (将所有 NUM_CLIENTS 相关的参数定义在一起) ---

# 定义其他固定参数 (根据你的代码片段)
NUM_CLIENTS = 20 # <--- Ensure this is set BEFORE dependent variables are created
BOOST_FACTORS = 2 # Re-declare or ensure it's defined once
MALICIOUS_RATIO = 0.3 # Re-declare or ensure it's defined once
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15 # Re-declare or ensure it's defined once
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
# --- 1. 实验扩展配置 ---

# 加载数据 (只需加载一次，放在循环外)
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)

# --- 2. 开始多维嵌套循环 ---
all_results = []
# --- 1. 实验扩展配置 ---
SEEDS = [1,2,3]
MECHANISM_LIST = ['MAB',  'FLAME','CosL2', 'TrimmedMean', 'Krum']
MECHANISM_LIST = ['FedAvg']
MAL_RATIOS = [ 0.3, 0.1, 0.2]
TOPO_TYPES = [ 'random_regular','scale_free']
DEFENSE_RATIOS = [ 0.2]  # 👈 新增：防御节点比例列表 (例如 10%, 20%, 30%)

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

# --- 2. 开始多维嵌套循环 ---


for topo_type in TOPO_TYPES:
    for mal_ratio in MAL_RATIOS:
        for def_ratio in DEFENSE_RATIOS: # 👈 第三层：防御比例循环

            # 动态计算当前配置
            num_mal = int(NUM_CLIENTS * mal_ratio)
            # 根据当前的 def_ratio 动态计算防御预算
            current_def_budget = int(NUM_CLIENTS * def_ratio)
            for seed_val in SEEDS:


                print(f"\n{'='*60}")
                print(f"⏰ 任务启动: {time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"📡 配置: Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                print(f"{'='*60}")
                print(f"\n{'#'*60}")
                print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                print(f"{'#'*60}")

                set_seed(seed_val)
                client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                G = generate_topology(NUM_CLIENTS, topo_type)
                neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}

                # 使用动态的 current_def_budget 分配节点
                malicious_clients, defense_nodes = allocate_malicious_nodes(
                    G, num_mal, current_def_budget, topology_type=topo_type,placement='Topology-Aware',# placement='Topology-Aware'
                )

                theo_intensities = calculate_theoretical_intensity(
                    neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                )

                for mech in MECHANISM_LIST:
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()
                    # 运行模拟
                    _, _, accs, asrs = run_simulation_CNN_GPU(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)

print(f"\n🎉 大型实验（含防御比例维度）全部完成！")


⏰ 任务启动: 2026-03-13 03:10:33
📡 配置: Topo=random_regular, Mal=0.3, Def=0.2

############################################################
📡 Topo: random_regular | Mal: 0.3 | Def: 0.2 | Seed: 1
############################################################
🚀 [System] Loading all client datasets directly into VRAM...
📸 [System] Extracting real validation data for S_Z Probe...
✅ Probe successfully extracted, shape: torch.Size([16, 3, 32, 32])

--- Round 1/15 ---
  [Attack Info] Estimated Benign Update Norm: 9.1473

--- Round 2/15 ---
  [Attack Info] Estimated Benign Update Norm: 8.6873

--- Round 3/15 ---
  [Attack Info] Estimated Benign Update Norm: 7.3884

--- Round 4/15 ---
  [Attack Info] Estimated Benign Update Norm: 6.2042

--- Round 5/15 ---
  [Attack Info] Estimated Benign Update Norm: 5.5281

--- Round 6/15 ---
  [Attack Info] Estimated Benign Update Norm: 5.0211

--- Round 7/15 ---
  [Attack Info] Estimated Benign Update Norm: 4.7581

--- Round 8/15 ---
  [Attack Info] Estimated Beni

KeyboardInterrupt: 

In [ ]:
# ... [前面的 import 部分保持不变] ...
import os
import time
import pandas as pd
import torch
import gc
from google.colab import drive
import numpy as np
import random
import pandas as pd
from data_loader import get_medical_data_augmented
from torch.utils.data import DataLoader, TensorDataset, random_split
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification,  get_linear_schedule_with_warmup
import glob # <--- Added this line
import sys
import time
from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from defense import get_high_value_defense_nodes
from main_cnn import run_simulation_CNN
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity
from data_loader import set_seed
import torch
import torch.nn as nn
import numpy as np
import copy
from torch.utils.data import DataLoader
from collections import defaultdict
import torch.nn.functional as F

from data_loader import set_seed
from MAB_fun import MABDefense_CNN,MABDefense
from MAB_fun import calculate_continuous_trust
from trainer import train_client_cnn
from defense import get_universal_stats,count_minimum_required_defenders
from defense import  calculate_krum_scores
from eval_DFL import evaluate_global_cnn
# ==========================================
# 1. CNN 专用辅助函数 (Sensitivity & Trapdoor)
# ==========================================
import sys
import os
NORM_FACTOR = 1
import os
# --- 1. 挂载 Google Drive (保持原位或提前) ---
drive_path = '/content/drive'
if not os.path.exists(drive_path):
    print("正在挂载 Google Drive...")
    drive.mount(drive_path)

# --- 2. 设置保存路径 (保持原位或提前) ---
# 结果将保存到 Drive 的这个文件夹下
SAVE_DIR = os.path.join(drive_path, 'MyDrive/FL_Experiments/cnn_Results_2026')
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

search_pattern = os.path.join(SAVE_DIR, f"Final_CNN_*.csv")
existing_files = glob.glob(search_pattern)
list_df = [pd.read_csv(f) for f in existing_files]

# FIX: Handle case where list_df is empty
if list_df:
    df_history = pd.concat(list_df, ignore_index=True)
else:
    df_history = pd.DataFrame() # Initialize as empty DataFrame if no files found
# ==========================================
# 1. 实验设置
# ==========================================
# 假设 functions2 和 SimpleCNN 已经定义好
# from functions2 import * # from simple_cnn import SimpleCNN

# 参数配置
# BOOST_FACTORS = 2 # This is used later, no need to move
# MALICIOUS_RATIO = 0.3 # This is used later, no need to move
GLOBAL_ROUNDS = 1 # This is used later, no need to move
intensity1 = 0.2
intensity2 = 0.2
import os
# --- 3. 实验配置: 定义核心参数 (将所有 NUM_CLIENTS 相关的参数定义在一起) ---

# 定义其他固定参数 (根据你的代码片段)
NUM_CLIENTS = 20 # <--- Ensure this is set BEFORE dependent variables are created
BOOST_FACTORS = 2 # Re-declare or ensure it's defined once
MALICIOUS_RATIO = 0.3 # Re-declare or ensure it's defined once
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15 # Re-declare or ensure it's defined once
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
# --- 1. 实验扩展配置 ---

# 加载数据 (只需加载一次，放在循环外)
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)

# --- 2. 开始多维嵌套循环 ---
all_results = []
# --- 1. 实验扩展配置 ---
SEEDS = [1,2,3]
MECHANISM_LIST = ['MAB',  'FLAME','CosL2', 'TrimmedMean', 'Krum']
MAL_RATIOS = [ 0.3, 0.4,0.1, 0.2]
TOPO_TYPES = [ 'random_regular','scale_free']
DEFENSE_RATIOS = [ 0.2]  # 👈 新增：防御节点比例列表 (例如 10%, 20%, 30%)

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

# --- 2. 开始多维嵌套循环 ---


for topo_type in TOPO_TYPES:
    for mal_ratio in MAL_RATIOS:
        for def_ratio in DEFENSE_RATIOS: # 👈 第三层：防御比例循环

            # 动态计算当前配置
            num_mal = int(NUM_CLIENTS * mal_ratio)
            # 根据当前的 def_ratio 动态计算防御预算
            current_def_budget = int(NUM_CLIENTS * def_ratio)
            for seed_val in SEEDS:


                print(f"\n{'='*60}")
                print(f"⏰ 任务启动: {time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"📡 配置: Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                print(f"{'='*60}")
                print(f"\n{'#'*60}")
                print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                print(f"{'#'*60}")

                set_seed(seed_val)
                client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                G = generate_topology(NUM_CLIENTS, topo_type)
                neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}

                # 使用动态的 current_def_budget 分配节点
                malicious_clients, defense_nodes = allocate_malicious_nodes(
                    G, num_mal, current_def_budget, topology_type=topo_type, placement='Topology-Aware'
                )

                theo_intensities = calculate_theoretical_intensity(
                    neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                )

                for mech in MECHANISM_LIST:
                    # 1. 先定义好文件名和路径
                    sys.stderr = logger

                    all_results = []
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    print(f"🚀 Running: {mech}")
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()
                    # 运行模拟
                    _, _, accs, asrs = run_simulation_CNN(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)
                    # --- 收集并保存结果 ---
                    result_entry_base = {
                        'seed': seed_val,
                        'mechanism': mech,
                        'malicious_ratio': mal_ratio,
                        'defense_ratio': def_ratio, # 👈 记录防御比例
                        'topology': topo_type,
                        'norm_factor': NORM_FACTOR,
                        'scale_factor': SCALE_FACTOR,
                        'global_rounds': GLOBAL_ROUNDS,
                        'start_time': start_wall_time,  # 👈 新增时间维度
                        'duration_sec': duration_sec    # 👈 新增耗时维度
                    }

                    for i in range(NUM_CLIENTS):
                        client_row = copy.deepcopy(result_entry_base)
                        client_row.update({
                            'client_id': i,
                            'final_acc': accs[i],
                            'final_asr': asrs[i],
                            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                        })
                        all_results.append(client_row)

                    # 实时保存
                    df_current = pd.DataFrame(all_results)

                    # 内存清理
                    torch.cuda.empty_cache()
                    gc.collect()
print(f"\n🎉 大型实验（含防御比例维度）全部完成！")